<a href="https://colab.research.google.com/github/Siddhartha127/Pytorch_Learning/blob/main/pytorch_02__autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

# 1. y= x**2

In [3]:
x=torch.tensor(3.0,requires_grad=True)


In [4]:
y=x**2

In [5]:
print(x,y)
#when we tell pytorch that we want to calculate gradient, then pytorch creates a computation graph internally

tensor(3., requires_grad=True) tensor(9., grad_fn=<PowBackward0>)


In [6]:
y.backward()

In [7]:
x.grad

tensor(6.)

# 2. y=x**2 , z=sin(y)

In [8]:
x=torch.tensor(3.0,requires_grad=True)

In [9]:
y=x**2

In [10]:
z=torch.sin(y)

In [14]:
print(x,y,z)

tensor(3., requires_grad=True) tensor(9., grad_fn=<PowBackward0>) tensor(0.4121, grad_fn=<SinBackward0>)


In [15]:
z.backward()

In [16]:
x.grad

tensor(-5.4668)

In [20]:
y.grad  #this will not be printed
'''
Conceptually, autograd keeps a record of data (tensors) & all executed operations (along with the resulting new tensors) in a directed acyclic graph (DAG) consisting of Function objects.
 In this DAG, leaves are the input tensors, roots are the output tensors.
 By tracing this graph from roots to leaves, you can automatically compute the gradients using the chain rule.'''
  https://pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html

<ipython-input-20-dd2913a88a42>:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at aten/src/ATen/core/TensorBody.h:489.)
  y.grad # this will not be printed


# Neural Network



```
x->W->sigmoid->y_pred->loss

input=cgpa
output=true or false

1. Linear Transformation: z=w.x+b
2. Activation Function(sigmoid): y_pred= σ(x)=1/1+e^-1
3. Loss function(Binary Cross-Entropy Loss): L=[y_target.ln(y_pred)+(1 - y_target).ln(1-y_pred)]

```




**Mannually**


```

derivative of y wrt to w and:  dL/dW

derivative of y wrt to b: dL/db
```





```
1. dL/dW = (dL/dy_pred) (dy_pred/dz) (dz/Dw)

dL/dW = {(y_pred -y)/y_pred(1-y_pred)} {y_pred(1-y_pred)} 1

dL/dW = (y_pred-y) * x

2. Similarly
dL/db = (y_pred-y) * 1
```



In [23]:
# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

In [24]:
print(x,y,w,b)

tensor(6.7000) tensor(0.) tensor(1.) tensor(0.)


In [25]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [26]:
# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [27]:
loss

tensor(6.7012)

In [28]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [29]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


**Using Autograd**

In [32]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [33]:
z = w*x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [34]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [35]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [36]:
loss.backward()

In [37]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


Vector inputs

In [40]:
x=torch.tensor([1.0,2.0,3.0], requires_grad=True)
x

tensor([1., 2., 3.], requires_grad=True)

In [42]:
y=(x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [43]:
y.backward()

In [44]:
x.grad

tensor([0.6667, 1.3333, 2.0000])



```
# clearing gradiants
# when we do multiple backward pass then the gradiants start to accumulate which is not good
#the gradiant is not cleared itself and when you run it again it is added to the new one
```



In [45]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [46]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [47]:
y.backward()

In [48]:
x.grad

tensor(4.)

In [49]:
x.grad.zero_()

tensor(0.)

In [50]:
# disable gradient tracking
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [51]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [52]:
y.backward()

In [53]:
x.grad

tensor(4.)

In [54]:
# option 1 - requires_grad_(False)
# option 2 - detach()
# option 3 - torch.no_grad()

In [55]:
x.requires_grad_(False)


tensor(2.)

In [56]:
x

tensor(2.)

In [57]:
y = x ** 2

In [58]:
y

tensor(4.)

In [59]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [60]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [61]:
z = x.detach()
z

tensor(2.)

In [62]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [63]:
y1 = z ** 2
y1

tensor(4.)

In [64]:
y.backward()

In [65]:
y1.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn